# TP 3 - Du système opérationnel au système décisionnel
### Spaceship Titanic · Architecture médaillon · EPSI

**Question :** comment transformer les décisions de qualité des TP 1 et 2 en un système de données partagé, consultable et réexécutable ?

```text
CSV sources → Bronze → Silver → Silver intégrée → Gold → SQL
```

**Objectifs :**
1. Distinguer l'espace de travail (*Workspace*) et les données (*Catalog Explorer*).
2. Conserver les sources dans Bronze, appliquer les traitements des TP 1 et 2 en Silver.
3. Intégrer deux sources par `PassengerId`, publier une table analytique Gold.
4. Vérifier les résultats et observer la traçabilité des tables.

## 0. Préparation

Dans **Workspace**, créez un dossier `EPSI/Spaceship` et importez ce notebook. Ce dossier contient du **code**, pas les tables.

Dans **Catalog Explorer**, créez un nouveau catalogue `spaceship`.

Exécutez **uniquement la cellule de préparation**, puis importez les deux CSV dans le volume `spaceship.landing.sources` :
- `spaceship_titanic_missing.csv`
- `passenger_biometrics.csv`

Les schémas et le volume sont créés automatiquement. Aucune configuration cloud particulière n'est nécessaire.

In [0]:
import os
from uuid import uuid4

import numpy as np
import pandas as pd
from pyspark.sql import functions as F, types as T

# Modifier cette valeur en fonction du nom de votre catalogue.
CATALOGUE = "spaceship"

for schema in ("landing", "bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOGUE}`.`{schema}`")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{CATALOGUE}`.`landing`.`sources`")

VOLUME = f"/Volumes/{CATALOGUE}/landing/sources"
CSV_PASSAGERS = f"{VOLUME}/spaceship_titanic_missing.csv"
CSV_BIOMETRIE = f"{VOLUME}/passenger_biometrics.csv"

BRONZE_PASSAGERS = f"{CATALOGUE}.bronze.passengers"
BRONZE_BIOMETRIE = f"{CATALOGUE}.bronze.biometrics"
SILVER_PASSAGERS = f"{CATALOGUE}.silver.passengers"
SILVER_BIOMETRIE = f"{CATALOGUE}.silver.biometrics"
SILVER_ENRICHIS = f"{CATALOGUE}.silver.passengers_enriched"
GOLD = f"{CATALOGUE}.gold.passenger_kpis_by_homeplanet"

print("Volume à ouvrir dans Catalog Explorer :", VOLUME)
print("Fichiers à déposer : spaceship_titanic_missing.csv et passenger_biometrics.csv")

Volume à ouvrir dans Catalog Explorer : /Volumes/spaceship/landing/sources
Fichiers à déposer : spaceship_titanic_missing.csv et passenger_biometrics.csv


## 1. Bronze - Conserver la donnée reçue

On lit les fichiers tels qu'ils arrivent : toutes les colonnes en texte, sans imputation ni correction. On ajoute des **métadonnées d'ingestion** : fichier source, date, lot et position de la ligne dans le CSV.

La table Bronze n'est créée **que si elle n'existe pas déjà** : relancer le notebook ne détruit pas l'ingestion précédente.

In [0]:

# Vérifier la présence des fichiers sources
for chemin in (CSV_PASSAGERS, CSV_BIOMETRIE):
    if not os.path.isfile(chemin):
        raise FileNotFoundError(
            f"Fichier introuvable : {chemin}. "
            "Importez les CSV dans le volume avant de continuer."
        )


def charger_bronze_si_absente(chemin, table, nom_source):
    # Ne pas réécrire une table Bronze existante
    if spark.catalog.tableExists(table):
        print(f"Bronze existante conservée : {table}")
        return

    # Lecture du CSV sans transformation métier
    brut = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(chemin)
    )

    # Contrôles simples sur les données sources
    assert brut.count() == 4000
    assert brut.select("PassengerId").distinct().count() == 4000

    # Ajout des métadonnées d'ingestion
    bronze = (
        brut
        .withColumn("_source_file", F.lit(nom_source))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_load_id", F.lit(str(uuid4())))
    )

    # Enregistrement dans une table Delta
    bronze.write.format("delta").mode("ignore").saveAsTable(table)
    print(f"Bronze créée : {table}")


# Chargement des deux sources
charger_bronze_si_absente(
    CSV_PASSAGERS,
    BRONZE_PASSAGERS,
    "spaceship_titanic_missing.csv"
)

charger_bronze_si_absente(
    CSV_BIOMETRIE,
    BRONZE_BIOMETRIE,
    "passenger_biometrics.csv"
)

# Vérification
print(
    "Lignes en Bronze :",
    spark.table(BRONZE_PASSAGERS).count(),
    "/",
    spark.table(BRONZE_BIOMETRIE).count()
)


Bronze créée : spaceship.bronze.passengers
Bronze créée : spaceship.bronze.biometrics
Lignes en Bronze : 4000 / 4000


In [0]:
# Afficher les 5 premières lignes de chaque table Bronze
display(spark.table(BRONZE_PASSAGERS).limit(5))
display(spark.table(BRONZE_BIOMETRIE).limit(5))

PassengerId,HomePlanet,CryoSleep,Deck,CabinSide,Destination,Age,VIP,RoomService,FoodCourt,Spa,_source_file,_ingested_at,_load_id
0001_01,Europa,False,B,P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e
0007_01,Earth,False,F,S,TRAPPIST-1e,35.0,False,0.0,785.0,216.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e
0009_01,Mars,False,F,P,TRAPPIST-1e,32.0,False,73.0,0.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e
0011_01,Earth,False,F,P,TRAPPIST-1e,28.0,False,8.0,974.0,2.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e
0014_01,Mars,False,F,P,55 Cancri e,27.0,False,null,122.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e


PassengerId,gender,height_cm,weight_kg,_source_file,_ingested_at,_load_id
1288_01,Female,164.2,61.3,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf
8115_02,Female,178.5,84.8,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf
1199_01,Female,167.2,66.7,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf
9097_01,Male,173.0,89.1,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf
6843_02,Male,175.0,89.1,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf


### Exercice 1 - Observer Bronze

Exécutez la requête. Vous devez retrouver les **200** `HomePlanet`, **320** `CryoSleep` et **280** `RoomService` manquants du TP 1.

Dans Catalog Explorer, ouvrez une table Bronze. Les types de `Age` et `RoomService` sont-ils déjà numériques ? Pourquoi ?

**Indice :** le choix `inferSchema=False` permet de différer le typage à Silver.

In [0]:
display(spark.sql(f"""
SELECT
  COUNT(*) AS passagers,
  SUM(CASE WHEN HomePlanet IS NULL THEN 1 ELSE 0 END) AS homeplanet_manquants,
  SUM(CASE WHEN CryoSleep IS NULL THEN 1 ELSE 0 END) AS cryosleep_manquants,
  SUM(CASE WHEN RoomService IS NULL THEN 1 ELSE 0 END) AS roomservice_manquants
FROM {BRONZE_PASSAGERS}
"""))

display(spark.sql(f"""
SELECT PassengerId, height_cm, weight_kg
FROM {BRONZE_BIOMETRIE}
WHERE CAST(height_cm AS DOUBLE) < 100 OR CAST(height_cm AS DOUBLE) > 230
   OR CAST(weight_kg AS DOUBLE) <= 0 OR CAST(weight_kg AS DOUBLE) > 250
ORDER BY PassengerId
"""))

passagers,homeplanet_manquants,cryosleep_manquants,roomservice_manquants
4000,200,320,280


PassengerId,height_cm,weight_kg
0787_01,1.72,70.8
4005_08,82.0,176.0
4050_01,152.0,820.0
6016_01,165.4,-74.0
7083_01,2250.0,76.9


> **À retenir - Bronze :** données proches de la source + métadonnées d'ingestion. Les anomalies n'y ont pas été « réparées ». On peut reconstruire les couches suivantes en cas de changement de règle.

## 2. Silver - Rejouer les TP 1 et 2

**Démonstration guidée**. Les traitements Python suivants sont fournis : vous avez déjà étudié ces méthodes dans les TP précédents. Ici, concentrez-vous sur le rôle de la couche Silver, les transformations réalisées et leur traçabilité.

Les traitements de Machine Learning utilisent scikit-learn. Pour cela, les données Spark sont converties en DataFrames pandas avec toPandas(), puis les résultats sont réintégrés dans Spark.

Cette approche convient à nos 4 000 observations, mais ne serait pas adaptée telle quelle à de très grands volumes de données. On utiliserait alors d’autres méthodes avec Spark MLlib par exemple.

### 2A. Silver `passengers` - Reprise du TP 1

- `HomePlanet` : **mode** ;
- `CryoSleep` : **KNN (5 voisins)** avec `Age`, `FoodCourt` et `Spa` (et standardisation) ;
- `RoomService` : **médiane (0)** et indicateur `RoomService_missing`.

Nous ajoutons deux indicateurs d'audit (`HomePlanet_imputed`, `CryoSleep_imputed`) ; ils ne changent pas les décisions du TP 1.

**Exécutez le code fourni ; ne reprogrammez pas le KNN.**

In [0]:

from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# 1. Imputation de CryoSleep avec KNN (TP 1)

# Lecture des données Bronze et conversion en pandas
entrainement = (
    spark.table(BRONZE_PASSAGERS)
    .select(
        "PassengerId",
        F.col("CryoSleep").cast("boolean"),
        F.col("Age").cast("double"),
        F.col("FoodCourt").cast("double"),
        F.col("Spa").cast("double"),
    )
    .orderBy("PassengerId")
    .toPandas()
)

variables = ["Age", "FoodCourt", "Spa"]
connus = entrainement["CryoSleep"].notna()

# Entraînement du KNN sur les observations connues
knn_cryo = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

knn_cryo.fit(
    entrainement.loc[connus, variables],
    entrainement.loc[connus, "CryoSleep"].astype(bool)
)

# Prédiction des valeurs manquantes
manquants = entrainement.loc[~connus, ["PassengerId"]].copy()

manquants["prediction"] = knn_cryo.predict(
    entrainement.loc[~connus, variables]
)

# Conversion des prédictions en DataFrame Spark
schema_predictions = T.StructType([
    T.StructField("PassengerId", T.StringType(), False),
    T.StructField("prediction", T.BooleanType(), False),
])

predictions_cryo = spark.createDataFrame(
    [
        (str(pid), bool(pred))
        for pid, pred in manquants[
            ["PassengerId", "prediction"]
        ].itertuples(index=False, name=None)
    ],
    schema=schema_predictions,
)


# 2. Typage des données Bronze

passagers_types = (
    spark.table(BRONZE_PASSAGERS)
    .select(
        "PassengerId",
        "HomePlanet",
        F.col("CryoSleep").cast("boolean"),
        "Deck",
        "CabinSide",
        "Destination",
        F.col("Age").cast("double"),
        F.col("VIP").cast("boolean"),
        F.col("RoomService").cast("double"),
        F.col("FoodCourt").cast("double"),
        F.col("Spa").cast("double"),
        "_source_file",
        "_ingested_at",
        "_load_id",
    )
)


# 3. Calcul du mode et de la médiane (TP 1)

mode_home = (
    passagers_types
    .where(F.col("HomePlanet").isNotNull())
    .groupBy("HomePlanet")
    .count()
    .orderBy(F.desc("count"), F.asc("HomePlanet"))
    .first()["HomePlanet"]
)

mediane_room = passagers_types.approxQuantile(
    "RoomService", [0.5], 0.0
)[0]


# 4. Construction de Silver et imputation

passagers_silver = (
    passagers_types
    # Conserver la trace des valeurs initialement manquantes
    .withColumn("HomePlanet_imputed", F.col("HomePlanet").isNull())
    .withColumn("CryoSleep_imputed", F.col("CryoSleep").isNull())
    .withColumn("RoomService_missing", F.col("RoomService").isNull().cast("int"))

    # Rattacher les prédictions KNN par PassengerId
    .join(predictions_cryo, "PassengerId", "left")

    # Appliquer les imputations
    .withColumn("HomePlanet", F.coalesce(F.col("HomePlanet"), F.lit(mode_home)))
    .withColumn("CryoSleep", F.coalesce(F.col("CryoSleep"), F.col("prediction")))
    .withColumn("RoomService", F.coalesce(F.col("RoomService"), F.lit(float(mediane_room))))
    .drop("prediction")
)


# 5. Contrôles et enregistrement dans Silver

assert passagers_silver.count() == 4000
assert passagers_silver.select("PassengerId").distinct().count() == 4000

passagers_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_PASSAGERS)

print(
    f"Silver passagers : {SILVER_PASSAGERS} | "
    f"mode={mode_home} | "
    f"médiane RoomService={mediane_room}"
)

display(spark.table(SILVER_PASSAGERS).limit(5))


Silver passagers : spaceship.silver.passengers | mode=Earth | médiane RoomService=0.0


PassengerId,HomePlanet,CryoSleep,Deck,CabinSide,Destination,Age,VIP,RoomService,FoodCourt,Spa,_source_file,_ingested_at,_load_id,HomePlanet_imputed,CryoSleep_imputed,RoomService_missing
0001_01,Europa,false,B,P,TRAPPIST-1e,39.0,false,0.0,0.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e,false,false,0
0007_01,Earth,false,F,S,TRAPPIST-1e,35.0,false,0.0,785.0,216.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e,false,false,0
0009_01,Mars,false,F,P,TRAPPIST-1e,32.0,false,73.0,0.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e,false,false,0
0011_01,Earth,false,F,P,TRAPPIST-1e,28.0,false,8.0,974.0,2.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e,false,false,0
0014_01,Mars,false,F,P,55 Cancri e,27.0,false,0.0,122.0,0.0,spaceship_titanic_missing.csv,2026-11-16T09:38:12.819Z,0a98162b-ef1f-4fd4-9500-a190ca19a45e,false,false,1


### 2B. Silver `biometrics` - Reprise du TP 2

Le code fourni applique **les mêmes décisions** que le TP 2 révisé :

1. Règles métier (5 lignes signalées), puis **3 corrections supposées** : inversion, mètres → centimètres, décimale.
2. Deux poids invalides mis à `NaN`, puis estimés par **KNNImputer** à 5 voisins, séparément par `gender`.
3. Détection des valeurs atypiques par **IQR global** et **Isolation Forest**.
4. Conservation des indicateurs `was_modified`, `weight_imputed`, `iqr_outlier`, `isolation_outlier`, `needs_review`.

**Exécutez le code fourni.** Aucun passager ne doit être supprimé.

In [0]:

import numpy as np
from sklearn.impute import KNNImputer
from sklearn.ensemble import IsolationForest

# 1. Lecture de Bronze et conversion en pandas

biometrie = (
    spark.table(BRONZE_BIOMETRIE)
    .select("PassengerId", "gender", "height_cm", "weight_kg")
    .withColumn("height_cm", F.col("height_cm").cast("double"))
    .withColumn("weight_kg", F.col("weight_kg").cast("double"))
    .orderBy("PassengerId")
    .toPandas()
)


# 2. Corrections ciblées issues du TP 2

df_clean = biometrie.copy()

# Taille et poids inversés
df_clean.loc[
    df_clean["PassengerId"] == "4005_08",
    ["height_cm", "weight_kg"]
] = [176, 82]

# Taille saisie en mètres au lieu de centimètres
df_clean.loc[
    df_clean["PassengerId"] == "0787_01", "height_cm"
] *= 100

# Erreur de décimale supposée
df_clean.loc[
    df_clean["PassengerId"] == "7083_01", "height_cm"
] /= 10


# 3. Poids invalides sans correction fiable

missing_weight = df_clean["PassengerId"].isin([
    "4050_01", "6016_01"
])

df_clean.loc[missing_weight, "weight_kg"] = np.nan


# 4. Traçabilité des corrections

df_clean["weight_imputed"] = missing_weight

df_clean["was_modified"] = df_clean["PassengerId"].isin([
    "4005_08", "0787_01", "7083_01",
    "4050_01", "6016_01"
])


# 5. Imputation des poids manquants avec KNN

variables_bio = ["height_cm", "weight_kg"]

for groupe in df_clean["gender"].unique():
    masque = df_clean["gender"] == groupe

    imputer = KNNImputer(n_neighbors=5, weights="distance")

    df_clean.loc[masque, variables_bio] = imputer.fit_transform(
        df_clean.loc[masque, variables_bio]
    )


# 6. Détection des valeurs atypiques avec IQR

q1 = df_clean[variables_bio].quantile(0.25)
q3 = df_clean[variables_bio].quantile(0.75)

iqr = q3 - q1
borne_basse = q1 - 1.5 * iqr
borne_haute = q3 + 1.5 * iqr

df_clean["iqr_outlier"] = (
    (df_clean[variables_bio] < borne_basse)
    | (df_clean[variables_bio] > borne_haute)
).any(axis=1)


# 7. Détection des valeurs atypiques avec Isolation Forest

modele_if = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

df_clean["isolation_outlier"] = (
    modele_if.fit_predict(df_clean[variables_bio]) == -1
)

# Marquer les observations à examiner
df_clean["needs_review"] = (
    df_clean["was_modified"]
    | df_clean["iqr_outlier"]
    | df_clean["isolation_outlier"]
)


# 8. Réintégration des corrections dans Spark

colonnes_corrigees = [
    "PassengerId",
    "height_cm",
    "weight_kg",
    "was_modified",
    "weight_imputed",
    "iqr_outlier",
    "isolation_outlier",
    "needs_review"
]

schema_corrige = T.StructType([
    T.StructField("PassengerId", T.StringType(), False),
    T.StructField("height_cm", T.DoubleType(), False),
    T.StructField("weight_kg", T.DoubleType(), False),
    *[
        T.StructField(c, T.BooleanType(), False)
        for c in (
            "was_modified",
            "weight_imputed",
            "iqr_outlier",
            "isolation_outlier",
            "needs_review"
        )
    ]
])

lignes_corrigees = [
    (
        str(pid),
        float(taille),
        float(poids),
        bool(mod),
        bool(imp),
        bool(iqr_flag),
        bool(if_flag),
        bool(revue)
    )
    for pid, taille, poids, mod, imp, iqr_flag, if_flag, revue
    in df_clean[colonnes_corrigees].itertuples(index=False, name=None)
]

corrections = spark.createDataFrame(
    lignes_corrigees,
    schema=schema_corrige
)

# Rattacher les corrections à Bronze par PassengerId
biometrie_silver = (
    spark.table(BRONZE_BIOMETRIE)
    .select(
        "PassengerId",
        "gender",
        "_source_file",
        "_ingested_at",
        "_load_id"
    )
    .join(corrections, "PassengerId", "inner")
)


# 9. Contrôles et enregistrement dans Silver

assert biometrie_silver.count() == 4000
assert biometrie_silver.select("PassengerId").distinct().count() == 4000

biometrie_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_BIOMETRIE)

# Résumé des traitements
print("Observations modifiées :", int(df_clean["was_modified"].sum()))
print("Poids imputés :", int(df_clean["weight_imputed"].sum()))
print("IQR :", int(df_clean["iqr_outlier"].sum()))
print("Isolation Forest :", int(df_clean["isolation_outlier"].sum()))
print("Needs review :", int(df_clean["needs_review"].sum()))

# Affichage de la table Silver
display(spark.table(SILVER_BIOMETRIE).limit(5))


Observations modifiées : 5
Poids imputés : 2
IQR : 5
Isolation Forest : 40
Needs review : 46


PassengerId,gender,_source_file,_ingested_at,_load_id,height_cm,weight_kg,was_modified,weight_imputed,iqr_outlier,isolation_outlier,needs_review
1288_01,Female,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf,164.2,61.3,false,false,false,false,false
8115_02,Female,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf,178.5,84.8,false,false,false,false,false
1199_01,Female,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf,167.2,66.7,false,false,false,false,false
9097_01,Male,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf,173.0,89.1,false,false,false,false,false
6843_02,Male,passenger_biometrics.csv,2026-11-16T09:38:19.505Z,18e66623-ed06-4a1b-94e5-94cc46053faf,175.0,89.1,false,false,false,false,false


### Exercice 2 - Comparer Bronze et Silver

**À faire :** exécutez les deux contrôles, puis retrouvez les chiffres des TP 1 et 2. Le jeu est-il « parfait » parce qu'il n'y a plus de valeur manquante dans les trois variables imputées ?

**Indice :** une valeur imputée reste **estimée**. `RoomService_missing` et `needs_review` permettent de garder une information sur l'incertitude.

In [0]:
display(spark.sql(f"""
SELECT COUNT(*) AS passagers,
       SUM(CASE WHEN HomePlanet IS NULL THEN 1 ELSE 0 END) AS homeplanet_manquants,
       SUM(CASE WHEN CryoSleep IS NULL THEN 1 ELSE 0 END) AS cryosleep_manquants,
       SUM(CASE WHEN RoomService IS NULL THEN 1 ELSE 0 END) AS roomservice_manquants,
       SUM(CAST(HomePlanet_imputed AS INT)) AS homeplanet_estimes,
       SUM(CAST(CryoSleep_imputed AS INT)) AS cryosleep_estimes,
       SUM(RoomService_missing) AS roomservice_initialement_manquants
FROM {SILVER_PASSAGERS}
"""))

display(spark.sql(f"""
SELECT COUNT(*) AS passagers,
       SUM(CAST(was_modified AS INT)) AS lignes_modifiees,
       SUM(CAST(weight_imputed AS INT)) AS poids_estimes,
       SUM(CAST(iqr_outlier AS INT)) AS atypiques_iqr,
       SUM(CAST(isolation_outlier AS INT)) AS atypiques_iforest,
       SUM(CAST(needs_review AS INT)) AS a_investiguer
FROM {SILVER_BIOMETRIE}
"""))

passagers,homeplanet_manquants,cryosleep_manquants,roomservice_manquants,homeplanet_estimes,cryosleep_estimes,roomservice_initialement_manquants
4000,0,0,0,200,320,280


passagers,lignes_modifiees,poids_estimes,atypiques_iqr,atypiques_iforest,a_investiguer
4000,5,2,5,40,46


> **À retenir - Silver :** typer, nettoyer, enrichir et **documenter l'incertitude**. Silver ne signifie pas « 100 % fiable ». Les règles métier peuvent changer : elles restent séparées de Bronze.

## 3. Intégrer les deux sources

Les fichiers ne contiennent pas les mêmes informations : `passengers` décrit le voyage, `biometrics` les mesures physiques. Leur clé commune est **`PassengerId`**.

**Démonstration :** créer `silver.passengers_enriched` par un `LEFT JOIN` : tous les passagers restent présents, avec leurs caractéristiques biométriques.

**À faire :** vérifiez qu'on obtient encore **4 000 lignes et 4 000 identifiants distincts**. Pourquoi tester les deux nombres plutôt que le seul nombre de lignes ?

**Indice :** une jointure peut multiplier les lignes quand la clé n'est pas unique dans la table jointe.

In [0]:
passagers = spark.table(SILVER_PASSAGERS)
biometrie = spark.table(SILVER_BIOMETRIE)
assert passagers.select("PassengerId").distinct().count() == 4000
assert biometrie.select("PassengerId").distinct().count() == 4000

enrichis = passagers.join(
    biometrie.select("PassengerId", "gender", "height_cm", "weight_kg",
                      "was_modified", "weight_imputed", "iqr_outlier",
                      "isolation_outlier", "needs_review"),
    on="PassengerId", how="left",
)
assert enrichis.count() == 4000
assert enrichis.select("PassengerId").distinct().count() == 4000

enrichis.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER_ENRICHIS)
display(spark.sql(f"""
SELECT COUNT(*) AS lignes, COUNT(DISTINCT PassengerId) AS identifiants_uniques,
       SUM(CASE WHEN gender IS NULL THEN 1 ELSE 0 END) AS sans_biometrie
FROM {SILVER_ENRICHIS}
"""))

lignes,identifiants_uniques,sans_biometrie
4000,4000,0


> **À retenir - Intégration :** Silver peut rassembler plusieurs systèmes pour fournir une référence cohérente. Il faut vérifier les clés et la cardinalité des jointures.

## 4. Gold - Publier un résultat exploitable

**Question métier :** combien de passagers viennent de chaque planète d'origine ? Quelle est leur taille moyenne ? Combien de profils sont à investiguer ?

Une table Gold est un **produit de données pour l'analyse**, pas nécessairement un schéma en étoile. La modélisation dimensionnelle viendra dans un autre chapitre.

**Exercice 3 :** exécutez la création de Gold. Puis essayez d'ajouter une colonne `roomservice_initialement_manquants` qui compte les valeurs `RoomService` imputées (utilisez `SUM(RoomService_missing)`).

**Indice :** ajoutez une **virgule** après `profiles_to_review`, puis une ligne `SUM(RoomService_missing) AS roomservice_initialement_manquants`. Les tables Silver restent intactes.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD} AS
SELECT
    HomePlanet,
    COUNT(*) AS passenger_count,
    ROUND(AVG(height_cm), 1) AS avg_height_cm,
    SUM(CAST(needs_review AS INT)) AS profiles_to_review,
    SUM(RoomService_missing) AS roomservice_initialement_manquants
FROM {SILVER_ENRICHIS}
GROUP BY HomePlanet
""")
display(spark.table(GOLD).orderBy("HomePlanet"))

HomePlanet,passenger_count,avg_height_cm,profiles_to_review,roomservice_initialement_manquants
Earth,2233,168.8,23,147
Europa,961,169.1,15,46
Mars,806,168.9,8,87


### Exercice 4 - Consommer Gold en SQL

Exécutez la requête, puis adaptez l'`ORDER BY` pour classer les planètes selon le nombre de profils `profiles_to_review` (au lieu du nombre de passagers).

In [0]:
display(spark.sql(f"""
SELECT HomePlanet, passenger_count, avg_height_cm, profiles_to_review
FROM {GOLD}
ORDER BY profiles_to_review DESC
"""))

HomePlanet,passenger_count,avg_height_cm,profiles_to_review
Earth,2233,168.8,23
Europa,961,169.1,15
Mars,806,168.9,8


## 5. Explorer Databricks et conclure

Dans **Catalog Explorer**, ouvrez `spaceship.gold.passenger_kpis_by_homeplanet` :

1. Observez ses colonnes et son contenu.
2. Ouvrez **Lineage** si l'onglet est disponible : remontez à `silver.passengers_enriched`, puis aux autres tables. Selon l'interface et les opérations, le lignage de certains calculs Python peut être partiel.
3. **Bonus** : exécutez `DESCRIBE HISTORY` pour consulter l'historique **technique** de cette table Delta (ce n'est **pas** l'historique métier des passagers).

**Bilan - quatre idées essentielles :**
- **Bronze** conserve l'état reçu et les informations d'ingestion.
- **Silver** applique les règles (TP 1 + TP 2), garde des indicateurs d'audit et intègre les sources.
- **Gold** publie des résultats cohérents pour les besoins d'analyse.
- Les transformations Silver et Gold peuvent être **rejouées à partir de Bronze** sans écraser les fichiers originaux.

**Suite du cours :** les notebooks des prochains chapitres pourront reprendre ces tables.

In [0]:
# Bonus facultatif : historique technique d'une table Delta.
display(spark.sql(f"DESCRIBE HISTORY {GOLD}"))

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-11-16T10:26:25.000Z,73244740279988,thibault.fb@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""60b3241d-e6fb-41c8-abd0-0ff47235d5e9"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-530e8019-b2bb-4992-ac60-ce48c6fb2b7f"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-52d0a572-3cd9-412e-9fa3-b41a8809c9c4""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1382403159715051),002a05ae-c326-439d-a8e1-0a32e0196e90,1116-092831-axm8iluo-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1966, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1965)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13
0,2026-11-13T11:15:02.000Z,73244740279988,thibault.fb@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""60b3241d-e6fb-41c8-abd0-0ff47235d5e9"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-530e8019-b2bb-4992-ac60-ce48c6fb2b7f"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-52d0a572-3cd9-412e-9fa3-b41a8809c9c4""}, statsOnLoad -> true, clusterByAuto -> true, clusteringOnWriteStatus -> Reason for skipping: Nonexistent clustering column, generally when replacing a Liquid table with a table without clustering columns)",null,List(1382403159715051),b012f0f7-6eb8-4555-86c5-2eaa9dcc4927,1113-105600-4pn7sd0j-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 1966)",null,Databricks-Runtime/19.14.x-aarch64-photon-scala2.13


---
### Repères perso

**Vérifications attendues, avec les CSV fournis et les paramètres du TP 2 :**
- Bronze : 4 000 lignes par table ; 200 / 320 / 280 valeurs manquantes pour `HomePlanet` / `CryoSleep` / `RoomService`.
- Silver `passengers` : aucune absence restante dans ces trois colonnes ; indicateurs d'imputation 200 / 320 / 280.
- Silver `biometrics` : 5 lignes initialement signalées, 2 poids imputés, 5 points IQR, 40 points Isolation Forest, **45** `needs_review` (avec ordre du CSV conservé et version scikit-learn comparable).
- Intégration : 4 000 lignes, 4 000 clés distinctes, 0 biométrie absente.
- Gold : une ligne par planète ; somme des passagers = 4 000, somme des alertes = 45.

**Réponses aux exercices :**
- Exercice 1 : les colonnes Bronze restent de type chaîne, car le typage métier est réservé à Silver.
- Exercice 2 : 0 valeur manquante dans les trois colonnes imputées ne garantit pas des données justes ; 200/320/280 imputations restent des estimations. La qualité est documentée, non décrétée.
- Intégration : tester `COUNT(*)` et `COUNT(DISTINCT PassengerId)` révèle une éventuelle multiplication des lignes. Ici les deux valent 4 000.
- Exercice 3 : une métrique `SUM(RoomService_missing)` retrouve les 280 valeurs initialement manquantes, ventilées par planète.
- Exercice 4 : le tri corrigé est `ORDER BY profiles_to_review DESC`. Gold fournit une définition analytique commune et évite à chaque équipe de recoder la même jointure et les mêmes traitements.

**Question de fond :** un modèle Gold peut publier un indicateur basé sur des valeurs estimées ; le fait que le pipeline soit reproductible ne rend pas automatiquement ces estimations exactes.

**Lignage :** les tables sont écrites depuis des DataFrames Spark dérivés des tables Bronze. Les calculs intermédiaires en pandas / scikit-learn ne sont pas retraçables au niveau colonne comme des expressions SQL simples.